# 10. Hình dự báo cho Chương 7

Đầu vào là chapter07_plot_inputs.zip hoặc hai ZIP kết quả của notebook 07 và 09. Đầu ra là 10_chapter07_figures.zip.

In [1]:
# Kiểm tra đầu vào
import hashlib
import io
import json
import os
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import FileLink, display
from matplotlib.ticker import FuncFormatter, MultipleLocator

KAGGLE_INPUT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
if not WORK_DIR.is_dir():
    WORK_DIR = Path.cwd()

INPUT_HASHES = {
    "final_test_predictions.csv": "23fb3f676803ae0fa6263b9e111dca6aa89566c1e56adaf56a33e5bba8aa2a00",
    "final_forecasts_sealed.csv": "f7fc9f946d0061d22933ad04fd30d5f4949da1a740b843e7c78765776aada208",
    "final_test_metrics.csv": "7051bef1a2d7dca20b79a6730f55e3fc7586286f3a6adc9c7daa71eeeb9c2cc6",
    "final_protocol_freeze.json": "9167a9d0675b28a51cef38439942d465779a43046bbfe023f202dc133d3ec169",
    "final_run_report.json": "e6dd7723d5b30d0a2b88bb567de42b1d1d44f22abca2d1ef62be1e301380bf2d",
    "xgb_feature_importance.csv": "0218bea5f8ee72bf0dd7c4deed76c75bd4734504a284e3512554ade54021ac4b",
}
INPUT_ZIPS = ("chapter07_plot_inputs.zip", "09_final_test_results.zip", "07_xgboost_results.zip")


def digest(data):
    return hashlib.sha256(data).hexdigest()


def load_input_files():
    files = {}
    for name in INPUT_HASHES:
        locations = [WORK_DIR / name]
        if KAGGLE_INPUT.is_dir():
            locations.extend(sorted(KAGGLE_INPUT.rglob(name)))
        for path in locations:
            if path.is_file():
                raw = path.read_bytes()
                if digest(raw) == INPUT_HASHES[name]:
                    files[name] = raw
                    break
    if files.keys() == INPUT_HASHES.keys():
        return files

    for zip_name in INPUT_ZIPS:
        archives = [WORK_DIR / zip_name]
        if KAGGLE_INPUT.is_dir():
            archives.extend(sorted(KAGGLE_INPUT.rglob(zip_name)))
        for path in archives:
            if not path.is_file():
                continue
            with ZipFile(path) as archive:
                for member in archive.namelist():
                    name = Path(member).name
                    if name in INPUT_HASHES and name not in files:
                        raw = archive.read(member)
                        if digest(raw) == INPUT_HASHES[name]:
                            files[name] = raw
            if files.keys() == INPUT_HASHES.keys():
                return files
    raise ValueError(
        "Thiếu đầu vào đã kiểm chứng từ notebook 07 hoặc 09. "
        "Thêm chapter07_plot_inputs.zip bản mới vào notebook."
    )


raw = load_input_files()
freeze = json.loads(raw["final_protocol_freeze.json"])
protocol = freeze["protocol"]
report = json.loads(raw["final_run_report.json"])
predictions = pd.read_csv(io.BytesIO(raw["final_test_predictions.csv"]), float_precision="round_trip")
sealed = pd.read_csv(io.BytesIO(raw["final_forecasts_sealed.csv"]), float_precision="round_trip")
metrics = pd.read_csv(io.BytesIO(raw["final_test_metrics.csv"]), float_precision="round_trip")

assert report["status"] == "PASS"
assert freeze["sha256"] == report["protocol_sha256"]
assert digest(raw["final_forecasts_sealed.csv"]) == report["forecast_sha256"]
assert len(predictions) == len(sealed) == report["forecast_rows"] == 336
assert len(metrics) == report["configuration_count"] == 12

key = ["config_id", "horizon_step"]
aligned = predictions.sort_values(key).reset_index(drop=True)
sealed_columns = ["config_id", "horizon_step", "d", "date", "model", "prediction"]
pd.testing.assert_frame_equal(
    aligned[sealed_columns],
    sealed.sort_values(key).reset_index(drop=True)[sealed_columns],
    check_exact=True,
)

days = np.arange(1, protocol["horizon"] + 1)
dates = pd.date_range(protocol["forecast_start"], protocol["forecast_end"]).strftime("%Y-%m-%d")
models = {model["config_id"]: model["model"] for model in protocol["models"]}
series = {config_id: rows for config_id, rows in aligned.groupby("config_id", sort=False)}
assert set(series) == set(models) == set(metrics["config_id"])
actual = next(iter(series.values()))["actual"].to_numpy()
for config_id, rows in series.items():
    np.testing.assert_array_equal(rows["horizon_step"], days)
    np.testing.assert_array_equal(rows["date"], dates)
    np.testing.assert_array_equal(rows["actual"], actual)
    assert rows["model"].unique().tolist() == [models[config_id]]
    assert np.isfinite(rows[["actual", "prediction"]].to_numpy()).all()

for row in metrics.itertuples(index=False):
    error = series[row.config_id]["prediction"].to_numpy() - actual
    rmse = np.sqrt(np.mean(error**2))
    np.testing.assert_allclose(
        [rmse / np.sqrt(report["rmsse_scale"]), np.mean(np.abs(error)), rmse],
        [row.rmsse, row.mae, row.rmse],
        rtol=1e-12,
    )

assert actual[20] == actual.max() == 59921 and dates[20] == "2016-05-15"
print("Đã kiểm tra 336 dự báo của 12 cấu hình và đối chiếu Bảng 10.")

importance = pd.read_csv(io.BytesIO(raw["xgb_feature_importance.csv"]), float_precision="round_trip")
selected = importance.loc[importance["selected_group"]]
assert len(selected) == 51 and selected["feature"].is_unique
assert selected["feature_group"].unique().tolist() == ["lags_calendar_events_snap"]
np.testing.assert_allclose(selected["normalized_gain"].sum(), 1)
top_features = selected.nlargest(10, "normalized_gain")
assert top_features["feature"].head(3).tolist() == ["lag_28", "lag_35", "lag_42"]
assert [f"{v:.4f}" for v in top_features["normalized_gain"].head(3)] == ["0.2542", "0.1128", "0.1118"]
print("Đã đối chiếu gain chuẩn hóa của XGBoost với Chương 7.")


Đã kiểm tra 336 dự báo của 12 cấu hình và đối chiếu Bảng 10.
Đã đối chiếu gain chuẩn hóa của XGBoost với Chương 7.


In [2]:
# Vẽ hình Chương 7
overview_ids = [*protocol["plot_models_selected_before_test"], "seasonal_naive_7"]
display_models = ("ARIMA", "ARIMAX", "SARIMA", "SARIMAX", "SVR", "XGBoost")
single_ids = {}
for model in display_models:
    config_id, = (key for key, name in models.items() if name == model)
    single_ids[model] = config_id

colors = {
    "ARIMA": "#504386", "ARIMAX": "#007C83", "SARIMA": "#825500",
    "SARIMAX": "#197443", "SVR": "#1463A3", "XGBoost": "#A94E07",
    "Seasonal Naive": "#A7344A",
}
styles = {"SVR": "--", "XGBoost": "-.", "SARIMAX": "-", "Seasonal Naive": ":"}
plot_ids = sorted(set(overview_ids) | set(single_ids.values()))
values = np.concatenate([actual, *[series[key]["prediction"].to_numpy() for key in plot_ids]])
ymin = np.floor((values.min() - 500) / 1000) * 1000
ymax = np.ceil((values.max() + 500) / 1000) * 1000

# Match the widths used by llncs.cls and the two-column figure in Chapter 7.
report_width = 12.2 / 2.54
panel_width = 0.49 * report_width
export_dpi = 900

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 7.4,
    "axes.labelsize": 7.4,
    "axes.linewidth": 0.55,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#697078",
    "axes.labelcolor": "#20262B",
    "text.color": "#20262B",
    "xtick.color": "#30363B",
    "ytick.color": "#30363B",
    "xtick.labelsize": 7.1,
    "ytick.labelsize": 7.1,
    "legend.fontsize": 7.2,
    "savefig.bbox": None,
    "savefig.facecolor": "white",
})
output_dir = WORK_DIR / "chapter07_figures"
output_dir.mkdir(exist_ok=True)
exports = {}


def setup_axis(ax, label_size, tick_size):
    ax.set_xlabel("Ngày trong tập Test", labelpad=2.5, fontsize=label_size)
    ax.set_ylabel("Nghìn sản phẩm/ngày", labelpad=2.5, fontsize=label_size)
    ax.set_xticks([1, 7, 14, 21, 28])
    ax.set_xlim(0.5, 28.5)
    ax.set_ylim(ymin, ymax)
    ax.tick_params(axis="both", length=2, width=0.5, pad=2, labelsize=tick_size)
    ax.yaxis.set_major_locator(MultipleLocator(5000))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value / 1000:.0f}"))
    ax.set_axisbelow(True)
    ax.grid(axis="y", color="#D7DEE5", linewidth=0.4)


def draw_actual(ax):
    ax.plot(days, actual, color="#20262B", linewidth=0.95,
            marker="o", markersize=1.65, label="Thực tế", zorder=3)


def add_legend(fig, ax, columns, font_size):
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.98),
               ncol=columns, frameon=False, fontsize=font_size, handlelength=1.6,
               handletextpad=0.4, columnspacing=1.0, borderaxespad=0)


def save_chart(fig, filename, source_name, config_ids=()):
    path = output_dir / filename
    fig.canvas.draw()
    bounds = fig.get_tightbbox(fig.canvas.get_renderer())
    width, height = fig.get_size_inches()
    assert 0 <= bounds.x0 < bounds.x1 <= width, (filename, bounds.bounds, width, height)
    assert 0 <= bounds.y0 < bounds.y1 <= height, (filename, bounds.bounds, width, height)
    fig.savefig(path, dpi=export_dpi)
    plt.close(fig)
    exports[filename] = {
        "source_file": source_name,
        "config_ids": list(config_ids),
        "size_inches": [round(float(width), 4), round(float(height), 4)],
        "sha256": digest(path.read_bytes()),
    }


fig, ax = plt.subplots(figsize=(report_width, report_width / 2))
fig.subplots_adjust(left=0.10, right=0.985, bottom=0.14, top=0.88)
setup_axis(ax, label_size=7.4, tick_size=7.1)
draw_actual(ax)
for config_id in overview_ids:
    model = models[config_id]
    label = "Seasonal Naive (7 ngày)" if model == "Seasonal Naive" else model
    ax.plot(days, series[config_id]["prediction"], color=colors[model],
            linestyle=styles[model], linewidth=0.9, label=label)
add_legend(fig, ax, columns=5, font_size=7.2)
save_chart(fig, "final_test_forecasts_vi.png", "final_test_predictions.csv", overview_ids)

for model, config_id in single_ids.items():
    fig, ax = plt.subplots(figsize=(panel_width, panel_width / 1.9))
    fig.subplots_adjust(left=0.14, right=0.98, bottom=0.23, top=0.845)
    setup_axis(ax, label_size=6.6, tick_size=6.4)
    draw_actual(ax)
    ax.plot(days, series[config_id]["prediction"], color=colors[model],
            linewidth=0.95, linestyle="--", label=f"Dự báo {model}")
    add_legend(fig, ax, columns=2, font_size=6.5)
    save_chart(fig, f"test_forecast_{model.lower()}.png", "final_test_predictions.csv", [config_id])

gain_width = 0.95 * report_width
fig, ax = plt.subplots(figsize=(gain_width, gain_width / 1.9))
fig.subplots_adjust(left=0.39, right=0.984, bottom=0.14, top=0.984)
bars = ax.barh(top_features["display_name"], top_features["normalized_gain"],
               color="#32735D", height=0.59, zorder=2)
ax.invert_yaxis()
ax.set_xlim(0, 0.30)
ax.set_xlabel("Gain trung bình chuẩn hóa", labelpad=2.5)
ax.spines["left"].set_visible(False)
ax.tick_params(axis="y", length=0, pad=4)
ax.tick_params(axis="x", length=2, width=0.5, pad=2)
for label in ax.get_yticklabels():
    label.set_fontfamily("DejaVu Sans Mono")
    label.set_fontsize(6.8)
ax.set_xticks([0, 0.05, 0.10, 0.15, 0.20, 0.25])
ax.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:.2f}".replace(".", ",")))
ax.grid(axis="x", color="#D7DEE5", linewidth=0.4)
ax.bar_label(bars, labels=[f"{v:.4f}".replace(".", ",") for v in top_features["normalized_gain"]],
             padding=2.2, fontsize=6.9)
save_chart(fig, "xgb_feature_importance_top10.png", "xgb_feature_importance.csv")


In [3]:
# Đóng gói ảnh
manifest = {
    "source": "09_final_test_results.zip",
    "feature_importance_source": "07_xgboost_results.zip",
    "protocol_sha256": freeze["sha256"],
    "source_sha256": INPUT_HASHES,
    "horizon": len(days),
    "start_date": dates[0],
    "end_date": dates[-1],
    "y_limits": [float(ymin), float(ymax)],
    "y_tick_display_divisor": 1000,
    "export_dpi": export_dpi,
    "figures": exports,
}
manifest_path = output_dir / "figure_manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
output_zip = WORK_DIR / "10_chapter07_figures.zip"
with ZipFile(output_zip, "w", compression=ZIP_DEFLATED) as archive:
    for filename in [*exports, manifest_path.name]:
        archive.write(output_dir / filename, filename)

assert len(exports) == 8
print("Đã xuất 8 ảnh và tệp đối chiếu trong 10_chapter07_figures.zip.")
display(FileLink(os.path.relpath(output_zip)))


Đã xuất 8 ảnh và tệp đối chiếu trong 10_chapter07_figures.zip.


/kaggle/working/10_chapter07_figures.zip